In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import time

# --- Configuration ---
CLEANED_DATA_PATH = '/content/drive/MyDrive/NetSci/food_data_processed_for_network_v2.csv' # Your cleaned data
K_NEIGHBORS = 10 # Number of nearest neighbors for each food
OUTPUT_GRAPH_FILE = '/content/drive/MyDrive/NetSci/food_knn_similarity_graph.gexf' # Gephi compatible format

# --- 1. Load and Prepare Data ---
print("--- Loading and Preparing Data ---")
try:
    df = pd.read_csv(CLEANED_DATA_PATH, index_col='name')
    print(f"Loaded data: {df.shape}")
except Exception as e:
    print(f"Error loading data: {e}")
    exit()

numeric_df = df.select_dtypes(include=np.number)
if numeric_df.empty:
    print("Error: No numeric features found.")
    exit()

print(f"Using {numeric_df.shape[1]} numeric features.")
scaler = StandardScaler()
normalized_data = scaler.fit_transform(numeric_df)
food_list = numeric_df.index.tolist()
print("Data normalized.")

# --- 2. Calculate Full Similarity Matrix ---
print("--- Calculating Full Cosine Similarity Matrix ---")
start_time = time.time()
similarity_matrix = cosine_similarity(normalized_data)
# Convert to DataFrame for easier indexing
similarity_df = pd.DataFrame(similarity_matrix, index=food_list, columns=food_list)
end_time = time.time()
print(f"Similarity matrix calculated in {end_time - start_time:.2f} seconds.")

# --- 3. Build k-NN Graph ---
print(f"--- Building k-NN Graph (k={K_NEIGHBORS}) ---")
start_time = time.time()
# Use a DiGraph because relationship isn't necessarily symmetric
G_knn = nx.DiGraph()

# Add all foods as nodes first (important for isolates if any)
for food in food_list:
    G_knn.add_node(food) # Add node attributes later if needed

edge_count = 0
for i, food_name in enumerate(food_list):
    # Get similarity scores for this food
    sim_scores = similarity_df[food_name]
    # Sort scores descending
    sorted_scores = sim_scores.sort_values(ascending=False)

    # Get top K neighbors (excluding self)
    # Ensure K_NEIGHBORS + 1 doesn't exceed length
    end_index = min(K_NEIGHBORS + 1, len(sorted_scores))
    # Exclude self (index 0)
    top_k_indices = sorted_scores.index[1:end_index]
    top_k_scores = sorted_scores.values[1:end_index]

    # Add edges from food_name to its top K neighbors
    for neighbor, score in zip(top_k_indices, top_k_scores):
        # Ensure score is a standard float for GEXF compatibility
        G_knn.add_edge(food_name, neighbor, weight=float(score))
        edge_count += 1

    if (i + 1) % 500 == 0: # Print progress
        print(f"Processed {i+1}/{len(food_list)} foods...")

end_time = time.time()
print(f"k-NN graph built in {end_time - start_time:.2f} seconds.")
print(f"Graph has {G_knn.number_of_nodes()} nodes and {edge_count} edges.") # Should be N * K if no ties

# --- 4. Save Graph ---
print(f"--- Saving k-NN Graph to {OUTPUT_GRAPH_FILE} ---")
try:
    nx.write_gexf(G_knn, OUTPUT_GRAPH_FILE)
    print("Graph saved successfully.")
    print(f"You can now load '{OUTPUT_GRAPH_FILE}' into Gephi or use it in your app.")
except Exception as e:
    print(f"Error saving graph: {e}")

--- Loading and Preparing Data ---
Loaded data: (8789, 74)
Using 74 numeric features.
Data normalized.
--- Calculating Full Cosine Similarity Matrix ---
Similarity matrix calculated in 0.72 seconds.
--- Building k-NN Graph (k=10) ---
Processed 500/8789 foods...
Processed 1000/8789 foods...
Processed 1500/8789 foods...
Processed 2000/8789 foods...
Processed 2500/8789 foods...
Processed 3000/8789 foods...
Processed 3500/8789 foods...
Processed 4000/8789 foods...
Processed 4500/8789 foods...
Processed 5000/8789 foods...
Processed 5500/8789 foods...
Processed 6000/8789 foods...
Processed 6500/8789 foods...
Processed 7000/8789 foods...
Processed 7500/8789 foods...
Processed 8000/8789 foods...
Processed 8500/8789 foods...
k-NN graph built in 10.58 seconds.
Graph has 8789 nodes and 87890 edges.
--- Saving k-NN Graph to /content/drive/MyDrive/NetSci/food_knn_similarity_graph.gexf ---
Graph saved successfully.
You can now load '/content/drive/MyDrive/NetSci/food_knn_similarity_graph.gexf' into 